# Visualization — earthlens fetches, digitalearth draws

earthlens owns no plotting layer. It fetches pixels from Earth Engine and
returns pyramids objects; every figure, map, animation and chart is produced by
[**digitalearth**](https://github.com/serapeum-org/Digital-Earth) over
cleopatra. This notebook walks the whole `earthlens → digitalearth` seam for the
GEE backend: resolving a dataset's default style from the catalog, rendering a
download client-side, the two Earth-Engine-side previews that need no download,
and a timelapse.

The plotting stack is an optional extra:

```bash
pip install earthlens[viz]      # or: uv sync --extra viz
```

## Setup

`pyramids` provides the `Dataset` that `earthlens` returns; `earthlens.gee`
provides the `GEE` backend and the `viz` helpers. The per-notebook output
directory holds the rasters each download writes.

In [ ]:
import os
from pathlib import Path

from earthlens.gee import GEE
from earthlens.gee.viz import style_for

OUT_DIR = Path("out") / "visualization"
OUT_DIR.mkdir(parents=True, exist_ok=True)

### Credentials

The GEE backend reads its service-account credentials from the
`GEE_SERVICE_ACCOUNT` / `GEE_SERVICE_KEY` environment variables. Both must be
set before running the download cells.

In [ ]:
SERVICE_ACCOUNT = os.environ["GEE_SERVICE_ACCOUNT"]
SERVICE_KEY = os.environ["GEE_SERVICE_KEY"]

## 1. Style hints from the catalog

`style_for` answers "how should this dataset be drawn?" from the catalog rather
than from a guess. It returns a frozen `StyleHints`, and its `source` says where
the answer came from:

* `"catalog"` — the publisher's own rendering, captured from the asset's STAC
  `gee:visualizations` (the band triple, stretch and palette Earth Engine's own
  catalog page uses). 894 of the 1104 curated datasets carry one.
* `"heuristic"` — derived from the band's declared `min` / `max` plus a
  colour-map match table, for a dataset the STAC gives no rendering.

No network or credentials are needed to resolve a style.

In [ ]:
rgb = style_for("COPERNICUS/S2_SR_HARMONIZED", ("B4", "B3", "B2"))
dem = style_for("USGS/SRTMGL1_003", ("elevation",))

print("Sentinel-2 true colour:", rgb.source, rgb.bands, "rgb =", rgb.is_rgb)
print("  per-band limits:", rgb.limits)
print("SRTM elevation:", dem.source, dem.bands)
print("  field kwargs for digitalearth:", dem.field_kwargs())

## 2. Render a downloaded raster, client-side

`GEE.visualize(result)` binds a download to its catalog style and delegates to
digitalearth. A single-band request goes through `quickmap`, stretched with the
hints' `vmin` / `vmax`. Here a single static SRTM tile over the Giza plateau.

In [ ]:
dem_gee = GEE(
    start="2000-02-11",
    end="2000-02-12",
    dataset="USGS/SRTMGL1_003",
    variables=["elevation"],
    aoi=[31.1, 29.9, 31.3, 30.1],
    cadence="raw",
    path=OUT_DIR,
    scale=200.0,
    reducer="mean",
    materialize=True,
)
dem_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
dem_result = dem_gee.download(progress_bar=False)

dem_viz = dem_gee.visualize(dem_result)
print("style source:", dem_viz.hints.source, "| kwargs:", dem_viz.hints.field_kwargs())
dem_map = dem_viz.still()
dem_map.fig

## 3. A true-colour composite

A three-band request is an RGB composite. `quickmap` has no composite renderer,
so the facade routes it to the tier's own `rgb_composite`, passing the catalog's
per-band `limits`. Here a monthly Sentinel-2 median over the same area.

In [ ]:
s2_gee = GEE(
    start="2021-06-01",
    end="2021-06-30",
    dataset="COPERNICUS/S2_SR_HARMONIZED",
    variables=["B4", "B3", "B2"],
    aoi=[31.1, 29.9, 31.25, 30.05],
    temporal_resolution="monthly",
    path=OUT_DIR,
    scale=100.0,
    reducer="median",
    materialize=True,
)
s2_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
s2_result = s2_gee.download(progress_bar=False)

s2_viz = s2_gee.visualize(s2_result)
print("RGB composite, limits:", s2_viz.hints.rgb_kwargs()["limits"][0])
s2_map = s2_viz.still()
s2_map.fig

## 4. Server-side previews — no download

Two previews are rendered by Earth Engine itself, before any pixels move, so
they need no `materialize` and no local render. `thumbnail` wraps `getThumbURL`
(a PNG URL Earth Engine renders on demand); `tiles` wraps `getMapId` to a
`{z}/{x}/{y}` XYZ template. Both are stretched by the same catalog hints.

In [ ]:
import ee

region = ee.Geometry.Rectangle([31.0, 29.9, 31.3, 30.2])
true_colour = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(region)
    .filterDate("2021-06-01", "2021-06-30")
    .median()
    .select(["B4", "B3", "B2"])
)

preview = s2_gee.visualize()  # no result needed for the EE-side previews
thumb_url = preview.thumbnail(true_colour, dimensions=400)
tile_source = preview.tiles(true_colour, name="S2 true colour")

print("thumbnail URL:", thumb_url[:72], "...")
print("tile template:", tile_source.url.split("/tiles/")[-1])

The thumbnail URL renders a real PNG when fetched — here it is inline.

In [ ]:
from IPython.display import Image

Image(url=thumb_url)

## 5. A timelapse

`timelapse` stacks a multi-date download into one animation in a single call to
digitalearth's collection-aware `animate`. Here three monthly means of daily
CHIRPS precipitation.

In [ ]:
chirps_gee = GEE(
    start="2021-06-01",
    end="2021-09-01",
    dataset="UCSB-CHG/CHIRPS/DAILY",
    variables=["precipitation"],
    aoi=[31.0, 30.0, 31.6, 30.6],
    temporal_resolution="monthly",
    path=OUT_DIR,
    scale=5000.0,
    reducer="mean",
    materialize=True,
)
chirps_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
chirps_result = chirps_gee.download(progress_bar=False)
print("frames:", chirps_result.collection().time_length)

gif_path = OUT_DIR / "chirps_summer.gif"
chirps_gee.visualize(chirps_result).timelapse(gif_path)
Image(filename=str(gif_path))

## 6. An Earth Engine tile layer on a slippy map

`tile_map` puts the Earth Engine tiles onto a digitalearth web or interactive
map — earthlens resolves the template inside Earth Engine, digitalearth draws
it, and nothing is downloaded. The slippy-map tiers are digitalearth's own
extras, so this needs `pip install 'digitalearth[web]'` (or `[interactive]`)
beyond `earthlens[viz]`; with those installed the call is:

```python
web_map = s2_gee.visualize().tile_map(true_colour, backend="web", name="S2")
web_map  # a MapLibre map with the Earth Engine layer
```

## What lives where

| Concern | Package |
|---|---|
| Fetching, catalogs, request shaping | earthlens |
| `getThumbURL` / `getMapId`, catalog style hints | earthlens (`earthlens.gee.viz`) |
| Maps, animations, charts, layout | digitalearth |
| Colour ramps, glyphs, the actual drawing | cleopatra |

## See also

* [Usage](../../reference/gee/usage.md) — the request shape and the object API.
* [Server-side computation](../../reference/gee/server-side-compute.md) — masks,
  filters, temporal helpers and the export sinks.